#### Web検索とLLMアプリを連動させる

In [1]:
# APIリクエストの準備 --------------
# 必要なモジュールをインポート
import os # OSの環境変数を読み取るために使う（APIキーの取得など） 
import json # jsonデータを便利に扱う
from dotenv import load_dotenv # .envから環境変数を読む（クライアント作るため）
from openai import OpenAI #  OpenAIのAPIを使ってLLMを操作する
from openai.types.chat import ChatCompletionToolParam # チャット補完で使うツールパラメータ型
from tavily import TavilyClient #検索APIを使うためのシンプルなクライアント
# TavilyClient でクライアントを作成してTavily Search APIを使用。

#環境変数の取得
load_dotenv("../.env")

# OpenI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

#tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# LLM のモデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数を作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})



TavilyClient() でTavily Search APIのクライアントを作成。

クライアント　=　APIにリクエストを送るための“窓口のひと”

.search() メソッドを付けて、窓口の人にquestionの内容の検索を頼んでいる感じ。

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://ekitan.com/event/station-2590',
   'title': '東京駅周辺のイベント - 駅探',
   'content': '## 東京駅のイベント一覧. ### 条件指定. + 東京駅／二重橋前駅／大手町駅(東京)駅. * #### （marunouchi）HOUSE COUNTDOWN PARTY 2025-2026. + 東京駅／二重橋前駅／大手町駅(東京)駅. 期間2025年12月11日(木)～12月13日(土) このイベントは終了しました. ### 近くの駅のイベント一覧. ### 東京駅沿線のイベント一覧. ### 東京駅の周辺情報. ### 東京駅の路線図・停車駅. ### 東京駅の運行情報. ### 東京駅発の人気の区間(乗換案内). ### 関連サービス. ### **おすすめ記事**.',
   'score': 0.7978881,
   'raw_content': None},
  {'url': 'https://rurubu.jp/andmore/area/130101',
   'title': 'るるぶ&more. - 東京駅周辺の記事・旅行ガイド・観光イベント情報',
   'content': '### 【最新】東京駅周辺のおいしいパスタ8選｜駅チカ・グランスタや丸の内、八重洲地下街のランチから、本格イタリアンまで！. ### 【シャングリ・ラ 東京】春の訪れを感じる、甘酸っぱいイチゴの「ストロベリーアフタヌーンティー」. ### 【東京駅】人気のお土産33選｜東京駅でしか買えない限定商品が多数！. ### 【東京駅】話題のドーナツショップ「ランディーズドーナツ」が東京ギフトパレット店をオープン！オープン記念限定ドーナツ＆グッズも. ### 丸の内・東京駅のイルミネーション特集2025-2026｜クリスマスにもおすすめ. ### King ＆ Prince登壇！ 丸の内「Celebration Tree」でMickey & Friends点灯式を開催【#編集部のおでかけキロク】. ### 【フォーシーズンズホテル東京大手町】ホリデーの華やぎを本格スイーツと感じる「フェスティブアフタヌーンティー」. ### 東京の真ん中で電車ビューに癒やされる

次は、ツール定義を関数化。

In [4]:
# ツールを定義（今回はパラメータが question だけのツール）

def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

言語モデルへの質問を行う関数を作る

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

ツール呼び出しが必要な場合の処理を行う関数をつくる

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "name": function_name, # ← 追加。これも必要
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

ユーザーからの質問を処理する関数をつくる

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

ここまでで準備が完了。

機能が関数化されているため、メインコードはシンプル。

まずはquestionを直接指定してテスト

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- 東京都の面積は約2,194平方キロメートルです。
- 沖縄県の面積は約2,271平方キロメートルです。

したがって、沖縄県が東京都よりも広いということになります。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内に東京駅で開催されるイベントに関する情報を以下にまとめました。

1. **東京駅イベント情報**  
   - **リンク**: [東京駅のイベントリスト](https://www.enjoytokyo.jp/event/list/area1306/)
   - **内容**: 様々なイベントが開催されており、2026年1月には「Happy New Year Tokyo」が行われる予定です。2025年11月から2026年2月にかけての詳細なイベント情報が掲載されています。

2. **2026年1月 東京のイベント20選**  
   - **リンク**: [2026年1月イベント情報](https://co-trip.jp/article/711482?page=2)
   - **内容**: 2026年1月に東京で開催されるイベントの情報がまとめられています。「ガウディ没後100年公式事業」など、注目のイベントが含まれています。

3. **東京駅の最新イベントまとめ**  
   - **リンク**: [東京駅最新イベントまとめ](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)
   - **内容**: 2ヶ月間にわたる多様なイベントが紹介されています。特に、12月下旬から1月初旬にかけてのイベントが多く取り上げられています。

これらのリンクから、具体的な日時やイベントの詳細をチェックすることができます。興味のあるイベントがあれば、ぜひ参加してみてください！


2種類チェック完了。ユーザーからの質問を受け付ける仕組みを作る。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは。'

こんにちは！何かお手伝いできることがありますか？


'質問:九州について教えて。'

九州は、日本の四つの主要な島の一つで、本州、四国、北海道と並ぶ位置にあります。九州の主な特徴を以下にまとめます。

1. **地理**: 九州は、北には福岡県、南には鹿児島県、東には宮崎県、西には長崎県があります。周りは海に囲まれており、特に東側には九州地方と大分県があります。そのため、美しい海岸線や温泉地が多いです。

2. **気候**: 九州は、温暖な気候が特徴で、特に南部は亜熱帯気候に属しています。夏は暑く、冬は比較的温暖なため、農業が盛んです。

3. **文化**: 九州には豊かな伝統文化があり、特に焼酎や博多ラーメン、佐賀の有田焼などが有名です。また、県ごとに異なる祭りや伝統行事も多く、観光スポットとしても人気があります。

4. **観光地**: 有名な観光地には、熊本城、太宰府天満宮、長崎のオランダ坂、指宿温泉などがあります。また、九州は自然も豊かで、阿蘇山や霧島連山などの山々も観光名所です。

5. **産業**: 九州は農業だけでなく、製造業や観光業も盛んです。また、近年ではIT産業やスタートアップ企業も増えてきています。

九州は自然、文化、食べ物が豊かで、訪れる人に多くの魅力を提供しています。興味があれば、より具体的な情報や特定の地域についての情報をお知らせください。

---ご利用ありがとうございました！---


キャラ設定してみる

In [12]:
# チャットボットへの組み込み
tools = define_tools()

# キャラ設定をroleに格納
role = "あなたは、明るくておもしろいタヌキです。関西弁を話し、絵文字を多用します。この設定を必ず守って！ 話し方はとてもフランク。"

# リストの最初(0番目)にsystemメッセージでキャラ設定をあらかじめいれる
messages=[
        {"role": "system", "content": role}, #0番目のメッセージ
        {"role": "user", "content": question} #1番目のメッセージ
    ]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除、ただし
    # キャラ設定がされている0番目は消さない
    if len(messages) > 8:
        del_message = messages.pop(1)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:関西について教えて。'

関西（かんさい）は、日本の地域の一つで、主に大阪府、京都府、兵庫県、奈良県、滋賀県、和歌山県から構成されています。以下は関西の特徴や見どころです。

1. **文化と歴史**: 
   - 京都は古都として知られ、多くの寺院や神社、伝統的な文化が残っています。金閣寺、清水寺、伏見稲荷大社などが有名です。また、京料理や和菓子も楽しめます。
   - 大阪は「食い倒れの街」として知られ、たこ焼き、お好み焼き、串カツなどの名物料理があります。

2. **観光地**:
   - 大阪城: かつての日本の重要な城で、美しい公園に囲まれています。
   - 奈良: 東大寺や奈良公園での鹿とのふれあいが人気です。
   - 和歌山: 高野山や白浜温泉など自然豊かな観光地があります。

3. **経済**:
   - 大阪は日本の商業の中心地として発展してきました。様々な企業の本社があります。

4. **イベント**:
   - 関西では一年を通じて多くの祭りやイベントが開催されます。特に春の桜や秋の紅葉は観光客に人気です。

5. **交通**:
   - 関西国際空港や新大阪駅など、交通の便が良く、国内外からのアクセスが便利です。

関西地方は、歴史的な名所と現代的な都市が融合した魅力溢れる地域です。興味があれば、特定のテーマや場所についてさらに詳しい情報をお伝えできます。


'質問:今日の関西のイベントは？'

今日の関西で開催されているイベント情報は以下の通りです。

1. [関西 今日におすすめのイベント](https://www.enjoytokyo.jp/event/list/regn02/its04/)
   - 内容: リンク先には、関西での様々なイベント情報が掲載されています。

2. [関西のイベント【今日2025年12月31日（水）】](https://www.walkerplus.com/event_list/today/ar0700/)
   - 内容: 今日開催されるイベントが194件掲載されています。詳細はリンク先でご確認ください。

3. [関西のイベント](https://www.walkerplus.com/event_list/ar0700/)
   - 内容: こちらも関西での様々なイベントが1178件紹介されています。

4. [関西・近郊 子ども・親子向け 今日のイベント情報 - いこーよ](https://iko-yo.net/events?region_ids%5B%5D=5&term=1)
   - 内容: 今日開催される子ども向けイベント情報を集めたページです。

5. [関西・近郊 予約不要（当日参加OK）のおでかけイベント情報 - いこーよ](https://iko-yo.net/events?region_ids%5B%5D=5&reservation_requireds%5B%5D=0)
   - 内容: 予約が不要なイベント情報の一覧です。

ぜひ、興味があるイベントをチェックしてみてください！

---ご利用ありがとうございました！---
